# 01 — Ingestion — Fisher's Iris

**Stage 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). Acquire the raw file, log its lineage and
provenance, validate it against a written schema, and record what governance applies — *before*
anything is cleaned, dropped, or analysed.

**Business question this project answers** (Stage 0, locked in `IMPLEMENTATION_PLAN.md`):

> Using only sepal and petal measurements, what is the simplest rule that separates the three Iris
> species — and, given 150 specimens, is the accuracy gap between competing models large enough to
> be measurable at all?

**What this notebook is looking for.** Not nulls. This file has none, and it is 90 years old and
150 rows long — hand-curated to a fault. The interesting question at ingestion is the opposite one:
*what does a file this clean still get wrong?*

Three specific things are checked here, and two of them contradict `DOCS/Iris_Dataset_Analysis.md`:

| # | Check | The notes say | This notebook tests |
|---|---|---|---|
| 1 | Is `Id` really information-free? | §2: "zero information value" | whether `Id` predicts `Species` |
| 2 | Are there duplicate rows? | §2: "no duplicates" | exact duplicates on the four measurements |
| 3 | Is every value plausible? | §1: "clean" | domain, range, granularity and internal-consistency rules |

Getting check 1 wrong is not a footnote. It is the difference between a harmless index and a column
that hands the model the answer.

In [1]:
import hashlib
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

SRC_NAME = "Iris.csv"
ROOT = Path.cwd() if (Path.cwd() / SRC_NAME).exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
for d in (RAW, PROC, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

MEASUREMENTS = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
TARGET = "Species"

print("project root :", ROOT.name)
print("source file  :", SRC_NAME)

project root : Iris Species
source file  : Iris.csv


## 1.1 Load, hash, and log the provenance chain

Immutability rule (STRUCTURE.md §1): the raw file is copied once, hashed, and never edited again.

Provenance matters more than usual here, because a finding later in this project depends on it.
This file did not come from Fisher. It came down a **four-step chain**, and errors were introduced
partway along it:

```
Anderson (1935)   collected 150 specimens, Gaspé Peninsula, Quebec
      ↓
Fisher (1936)     published the table in "The Use of Multiple Measurements in Taxonomic Problems"
      ↓
UCI ML Repository digitised it — and its own documentation records two erroneous rows
      ↓
Kaggle Iris.csv   this file, with an added `Id` column
```

The UCI errata note states that the 35th sample should read `4.9,3.1,1.5,0.2` (error in the 4th
feature) and the 38th sample should read `4.9,3.6,1.4,0.1` (errors in the 2nd and 3rd features).
Whether *this* file carries those errors is tested in `02_cleaning.ipynb` — the chain is logged here
so the test has something to be checked against.

In [2]:
src_path = ROOT / SRC_NAME
raw_bytes = src_path.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()

df = pd.read_csv(src_path)

lineage = pd.DataFrame([{
    "artefact": "raw_source",
    "source_file": SRC_NAME,
    "collector": "Edgar Anderson (1935), Gaspe Peninsula, Quebec",
    "publisher": "R.A. Fisher (1936) -> UCI ML Repository -> Kaggle",
    "known_errata": "UCI documents 2 erroneous rows (35, 38) - tested in 02_cleaning",
    "sha256": sha256,
    "bytes": len(raw_bytes),
    "pulled_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "n_rows": df.shape[0],
    "n_cols": df.shape[1],
    "delimiter": ",",
    "header": "row 0",
    "encoding": "utf-8",
}])
lineage.to_csv(RAW / "_lineage.csv", index=False)
df.to_csv(RAW / "Iris_raw_immutable.csv", index=False)

print(f"shape        : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"sha256       : {sha256}")
print(f"dtypes       : {dict(df.dtypes.astype(str).value_counts())}")
print(f"memory       : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"columns      : {list(df.columns)}")

shape        : 150 rows x 6 columns
sha256       : 600ac44f23c2e6e0ae37daac8ceb2baba4df963efa580eb31b3b576b28e34c55
dtypes       : {'float64': np.int64(4), 'int64': np.int64(1), 'str': np.int64(1)}
memory       : 9.1 KB
columns      : ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']


In [3]:
df.head(5)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 1.2 Schema validation

One row per rule, pass/fail, written to `data/raw/_schema_validation.csv`. Four families of rule:

- **Domain** — is every value in the documented set of allowed values?
- **Plausibility** — is every value in a physically sensible range?
- **Granularity** — at what precision was this actually measured?
- **Internal consistency** — does the morphology obey the arithmetic it logically must?

The granularity family is the one that earns its keep on a file with no nulls. It is not cosmetic:
it bounds how finely any threshold rule can be cut, and it explains why some specimens are
genuinely unresolvable rather than merely hard.

In [4]:
rules = []

def check(family, rule, ok, detail=""):
    rules.append({"family": family, "rule": rule, "result": "PASS" if ok else "FAIL", "detail": detail})

# --- Domain -------------------------------------------------------------
expected_species = {"Iris-setosa", "Iris-versicolor", "Iris-virginica"}
found_species = set(df[TARGET].astype(str).unique())
check("domain", "Species in {setosa, versicolor, virginica}",
      found_species == expected_species, f"found: {sorted(found_species)}")

check("domain", "Species counts are 50/50/50",
      df[TARGET].value_counts().nunique() == 1 and df[TARGET].value_counts().iloc[0] == 50,
      str(df[TARGET].value_counts().to_dict()))

check("domain", "Id is unique", df["Id"].is_unique, f"{df['Id'].nunique()} unique of {len(df)}")
check("domain", "Id spans 1..150 with no gaps",
      df["Id"].min() == 1 and df["Id"].max() == 150 and len(df) == 150,
      f"min={df['Id'].min()}, max={df['Id'].max()}")
check("domain", "all measurements strictly positive",
      bool((df[MEASUREMENTS] > 0).all().all()), "")

# --- Plausibility -------------------------------------------------------
ranges = {"SepalLengthCm": (4.0, 8.0), "SepalWidthCm": (1.9, 4.5),
          "PetalLengthCm": (0.9, 7.0), "PetalWidthCm": (0.05, 2.6)}
for col, (lo, hi) in ranges.items():
    obs_lo, obs_hi = df[col].min(), df[col].max()
    check("plausibility", f"{col} within [{lo}, {hi}] cm",
          bool(obs_lo >= lo and obs_hi <= hi), f"observed [{obs_lo}, {obs_hi}]")

# --- Granularity --------------------------------------------------------
for col in MEASUREMENTS:
    tenths = np.round(df[col] * 10).astype(int)
    on_grid = bool(np.allclose(df[col], tenths / 10))
    n_unique = df[col].nunique()
    check("granularity", f"{col} recorded to 0.1 cm", on_grid,
          f"{n_unique} distinct values across {len(df)} rows")

# --- Internal consistency ----------------------------------------------
check("consistency", "PetalLength > PetalWidth for every specimen",
      bool((df["PetalLengthCm"] > df["PetalWidthCm"]).all()),
      f"violations: {int((df['PetalLengthCm'] <= df['PetalWidthCm']).sum())}")
check("consistency", "SepalLength > SepalWidth for every specimen",
      bool((df["SepalLengthCm"] > df["SepalWidthCm"]).all()),
      f"violations: {int((df['SepalLengthCm'] <= df['SepalWidthCm']).sum())}")
check("consistency", "no nulls anywhere",
      int(df.isnull().sum().sum()) == 0, f"total nulls: {int(df.isnull().sum().sum())}")

schema = pd.DataFrame(rules)
schema.to_csv(RAW / "_schema_validation.csv", index=False)

print(f"{(schema.result == 'PASS').sum()} of {len(schema)} rules PASS\n")
schema

16 of 16 rules PASS



,family,rule,result,detail
0,domain,"Species in {setosa, versicolor, virginica}",PASS,"found: ['Iris-setosa', 'Iris-versicolor', 'Iri..."
1,domain,Species counts are 50/50/50,PASS,"{'Iris-setosa': 50, 'Iris-versicolor': 50, 'Ir..."
2,domain,Id is unique,PASS,150 unique of 150
3,domain,Id spans 1..150 with no gaps,PASS,"min=1, max=150"
4,domain,all measurements strictly positive,PASS,
5,plausibility,"SepalLengthCm within [4.0, 8.0] cm",PASS,"observed [4.3, 7.9]"
6,plausibility,"SepalWidthCm within [1.9, 4.5] cm",PASS,"observed [2.0, 4.4]"
7,plausibility,"PetalLengthCm within [0.9, 7.0] cm",PASS,"observed [1.0, 6.9]"
8,plausibility,"PetalWidthCm within [0.05, 2.6] cm",PASS,"observed [0.1, 2.5]"
9,granularity,SepalLengthCm recorded to 0.1 cm,PASS,35 distinct values across 150 rows


**Every rule passes.** That is worth pausing on rather than celebrating.

Null-counting and range-checking cannot see the two defects this file actually has. A transcription
error that replaces one valid measurement with another valid measurement passes every rule above.
So does a duplicated row. So does an identifier that encodes the answer. §1.3–§1.4 go looking for
exactly those.

The granularity result is the substantive finding of this section: **every measurement sits on a
0.1 cm grid**, and `PetalWidthCm` has only 22 distinct values across 150 specimens. Two flowers of
different species can therefore record *identical* measurements — not because the data is wrong, but
because the ruler ran out of resolution. That sets a hard ceiling on any classifier, and it is not
a ceiling more data or a better model can lift.

In [5]:
gran = pd.DataFrame({
    "distinct_values": df[MEASUREMENTS].nunique(),
    "min": df[MEASUREMENTS].min(),
    "max": df[MEASUREMENTS].max(),
    "range_cm": df[MEASUREMENTS].max() - df[MEASUREMENTS].min(),
})
gran["possible_grid_points"] = (gran["range_cm"] * 10).round().astype(int) + 1
gran["grid_coverage"] = (gran["distinct_values"] / gran["possible_grid_points"]).round(3)
gran

,distinct_values,min,max,range_cm,possible_grid_points,grid_coverage
SepalLengthCm,35,4.3,7.9,3.6,37,0.946
SepalWidthCm,23,2.0,4.4,2.4,25,0.920
PetalLengthCm,43,1.0,6.9,5.9,60,0.717
PetalWidthCm,22,0.1,2.5,2.4,25,0.880


## 1.3 The `Id` forensic — the most consequential check in this notebook

`DOCS/Iris_Dataset_Analysis.md` §2 describes `Id` as a *"pure row index (1–150), sequential, zero
information value — an artifact of the CSV export, not a real feature"*, and §6 warns against using
it because it is *"meaningless signal"*.

The first half of that is right and the second half is exactly backwards. An index is only
information-free if it is **unrelated to the label**. This one is not: the file is sorted by
species, so `Id` does not merely correlate with `Species` — it *determines* it.

Two tests. First, where does each species live in the ID range?

In [6]:
id_blocks = (df.groupby(TARGET)["Id"]
               .agg(n="size", id_min="min", id_max="max")
               .assign(contiguous=lambda d: (d.id_max - d.id_min + 1) == d.n))
print(id_blocks.to_string())
print()
print("species change points in row order:",
      [int(i) for i in df.index[df[TARGET].astype(str).ne(df[TARGET].astype(str).shift())][1:]])
print("=> the file is sorted by species; each class is one unbroken block of 50 IDs")

                  n  id_min  id_max  contiguous
Species                                        
Iris-setosa      50       1      50        True
Iris-versicolor  50      51     100        True
Iris-virginica   50     101     150        True

species change points in row order: [50, 100]
=> the file is sorted by species; each class is one unbroken block of 50 IDs


Second, and decisively: hand `Id` to a model **as its only feature** and see what it scores under
ordinary shuffled cross-validation — the exact protocol a standard workflow would use.

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

y = np.asarray(df[TARGET].astype(str))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=RANDOM_STATE)

id_only = cross_val_score(DecisionTreeClassifier(random_state=RANDOM_STATE),
                          df[["Id"]].to_numpy(float), y, cv=cv)
petals = cross_val_score(DecisionTreeClassifier(random_state=RANDOM_STATE),
                         df[["PetalLengthCm", "PetalWidthCm"]].to_numpy(float), y, cv=cv)
majority = cross_val_score(DummyClassifier(strategy="most_frequent"),
                           df[["Id"]].to_numpy(float), y, cv=cv)

leak = pd.DataFrame({
    "feature set": ["Id alone (the 'meaningless' index)",
                    "PetalLength + PetalWidth (real morphology)",
                    "majority class (no information)"],
    "cv_accuracy": [id_only.mean(), petals.mean(), majority.mean()],
    "cv_sd": [id_only.std(), petals.std(), majority.std()],
}).round(4)

print(leak.to_string(index=False))
print()
print(f"Id alone scores {id_only.mean():.1%} — it out-performs the two real petal measurements "
      f"({petals.mean():.1%}).")

                               feature set  cv_accuracy  cv_sd
        Id alone (the 'meaningless' index)       0.9870 0.0182
PetalLength + PetalWidth (real morphology)       0.9467 0.0327
           majority class (no information)       0.3333 0.0000

Id alone scores 98.7% — it out-performs the two real petal measurements (94.7%).


**So What:** `Id` is the single strongest predictor in this file, and it is pure leakage. A model
built on it would report near-perfect accuracy and would classify a *new* flower at chance. The
notes' claim that `Id` carries "zero information value" is the precise belief that makes the common
idiom `X = df.drop(columns=['Species'])` feel safe — and here that idiom produces a 98.7% model that
knows nothing about irises.

**Implication:** this is not a quirk of a teaching dataset. It is the same failure as a production
export sorted by status, region, or churn flag — the most common silent leak in tabular work, and it
survives every null-check and range-check in §1.2. `Id` is set as the index below and **never enters
a feature matrix again**; the one exhibit above is the only use it gets.

In [8]:
dup_measure = int(df.duplicated(subset=MEASUREMENTS).sum())
dup_full = int(df.duplicated(subset=MEASUREMENTS + [TARGET]).sum())

print(f"duplicate Id values                        : {int(df['Id'].duplicated().sum())}")
print(f"duplicate rows on the 4 measurements       : {dup_measure}")
print(f"duplicate rows on measurements + species   : {dup_full}")
print()
print("The notes (DOCS §2) state 'no duplicates'. Contradicted — the rows involved:")
df[df.duplicated(subset=MEASUREMENTS, keep=False)].sort_values(MEASUREMENTS)

duplicate Id values                        : 0
duplicate rows on the 4 measurements       : 3
duplicate rows on measurements + species   : 3

The notes (DOCS §2) state 'no duplicates'. Contradicted — the rows involved:


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
9,10,4.9,3.1,1.5,0.1,Iris-setosa
34,35,4.9,3.1,1.5,0.1,Iris-setosa
37,38,4.9,3.1,1.5,0.1,Iris-setosa
101,102,5.8,2.7,5.1,1.9,Iris-virginica
142,143,5.8,2.7,5.1,1.9,Iris-virginica


## 1.4 What ingestion found

| Check | Notes claim | Verified result | Verdict |
|---|---|---|---|
| Nulls | none | none | notes correct |
| Summary statistics | §5 table | reproduces exactly (§1.5) | notes correct |
| Class balance | 50/50/50 | 50/50/50 | notes correct |
| **`Id` information** | "zero information value" | **98.7% CV accuracy alone** | **notes wrong — it is leakage** |
| **Duplicates** | "no duplicates" | **3 duplicate rows** | **notes wrong** |
| Value-level correctness | "clean" | deferred to `02_cleaning` | pending |

Two of six claims fail. Both failures are invisible to null-counting, which is the transferable
point: *"0 missing values"* is a statement about completeness, not about correctness.

In [9]:
summary = df[MEASUREMENTS].describe().T[["mean", "std", "min", "max"]].round(2)
notes_table = pd.DataFrame(
    {"notes_mean": [5.84, 3.05, 3.76, 1.20], "notes_std": [0.83, 0.43, 1.76, 0.76],
     "notes_min": [4.3, 2.0, 1.0, 0.1], "notes_max": [7.9, 4.4, 6.9, 2.5]},
    index=MEASUREMENTS)
cmp = summary.join(notes_table)
cmp["matches_notes"] = (
    np.isclose(cmp["mean"], cmp["notes_mean"], atol=0.01)
    & np.isclose(cmp["std"], cmp["notes_std"], atol=0.01)
    & np.isclose(cmp["min"], cmp["notes_min"]) & np.isclose(cmp["max"], cmp["notes_max"]))
print("DOCS/Iris_Dataset_Analysis.md section 5 summary table, re-derived:\n")
cmp

DOCS/Iris_Dataset_Analysis.md section 5 summary table, re-derived:



,mean,std,min,max,notes_mean,notes_std,notes_min,notes_max,matches_notes
SepalLengthCm,5.84,0.83,4.3,7.9,5.84,0.83,4.3,7.9,True
SepalWidthCm,3.05,0.43,2.0,4.4,3.05,0.43,2.0,4.4,True
PetalLengthCm,3.76,1.76,1.0,6.9,3.76,1.76,1.0,6.9,True
PetalWidthCm,1.20,0.76,0.1,2.5,1.20,0.76,0.1,2.5,True


## 1.5 Governance

Botanical measurements of plants collected in 1935. No persons, no PII, no consent surface, public
domain. Safe to commit in full.

**The fairness audit is recorded as not-applicable here, with its reason**, so the cross-cutting
gate has a documented answer rather than a blank. STRUCTURE.md makes a fairness audit mandatory for
any model affecting people; this model classifies flowers. No protected attribute exists in the
data and no person is scored by the output.

**One bias check does apply and is not waived: representation.** All 150 specimens come from a
single collection — one collector, one peninsula, one season, and per Fisher's own paper two of the
three species were taken from the same pasture on the same day, measured by the same person with the
same instrument. That is a **sampling frame, not a population**. Nothing in this project licenses a
claim about *Iris* as a genus; every finding is scoped to these 150 specimens.

In [10]:
governance = pd.DataFrame([
    {"item": "PII present", "status": "No",
     "detail": "botanical measurements only; no persons in the data"},
    {"item": "Data classification", "status": "Public",
     "detail": "public domain since 1936; safe to commit"},
    {"item": "Consent / purpose limitation", "status": "N/A",
     "detail": "no personal data subject exists"},
    {"item": "Fairness audit required", "status": "No - with reason",
     "detail": "no protected attribute in data; output scores flowers, not people (STRUCTURE.md cross-cutting)"},
    {"item": "Representation bias check", "status": "REQUIRED - executed",
     "detail": "single collector / site / season, 1935 Gaspe Peninsula; sample not population"},
    {"item": "Retention", "status": "N/A", "detail": "public dataset, no retention obligation"},
    {"item": "Deployment ceiling", "status": "None claimed",
     "detail": "90-year-old single-site sample cannot support a deployed taxonomic classifier"},
])
governance.to_csv(RAW / "_governance.csv", index=False)
governance

,item,status,detail
0,PII present,No,botanical measurements only; no persons in the...
1,Data classification,Public,public domain since 1936; safe to commit
2,Consent / purpose limitation,N/A,no personal data subject exists
3,Fairness audit required,No - with reason,no protected attribute in data; output scores ...
4,Representation bias check,REQUIRED - executed,"single collector / site / season, 1935 Gaspe P..."
5,Retention,N/A,"public dataset, no retention obligation"
6,Deployment ceiling,None claimed,90-year-old single-site sample cannot support ...


## Stage 1 — Gate Checklist

- [x] **Raw data saved to `data/raw/`** (untouched copy + SHA-256 hash)
- [x] **Source, timestamp, and row/column counts logged** — `_lineage.csv`, including the
      four-step provenance chain the errata test in `02_cleaning` depends on
- [x] **Schema matches expectations** — 15 rules across domain / plausibility / granularity /
      internal-consistency, all PASS, written to `_schema_validation.csv`; the granularity finding
      (0.1 cm grid, 22 distinct petal-width values) is carried forward as a hard modelling ceiling
- [x] **PII / data governance review completed** — `_governance.csv`; fairness audit answered
      *not-applicable with reason*, representation-bias check flagged as required and executed
- [x] **Two claims in `DOCS/Iris_Dataset_Analysis.md` tested and refuted** — `Id` is not
      information-free (98.7% CV accuracy alone), and the file is not duplicate-free (3 rows)

**Next:** `02_cleaning.ipynb` — diagnose the 3 duplicate rows, test this file against the UCI
errata, and fix the measurement-scale contract.